<a href="https://colab.research.google.com/github/BenCheung1/Pytorch_Sandbox/blob/main/AIML_DSB_SupervisedML_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Import Torch libraries
import torch
import torch.nn as nn
import torch.optim as optim

# Import Python essential libraries
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# https://www.datasciencebase.com/supervised-ml/algorithms/support-vector-machines/pytorch-example/
# Load the Iris dataset
iris = datasets.load_iris()
data = pd.DataFrame(data=iris.data, columns=iris.feature_names)
data['target'] = iris.target

# Select only two classes for binary classification (Setosa and Versicolor)
data = data[data['target'] != 2]

# [1] Split the data into features (X) and target (y)
X = data.drop('target', axis=1).values
y = data['target'].values

# [2] Split the data into Training and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# [3] Standardize the features (SVM is sensitive to feature scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# [4] Convert the data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # Add extra dimension for binary classification
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
# Reduce to a binary classification problem

In [11]:
# CREATE THE SUPPORT VECTOR MACHINE MODEL
class SVM(nn.Module):
    def __init__(self, input_size):
        super(SVM, self).__init__()
        self.linear = nn.Linear(input_size, 1)  # One output for binary classification

    # Apply a Linear Transform
    def forward(self, x):
        return self.linear(x)  # Linear combination w^T x + b

# Initialize the model
model = SVM(input_size=X_train_tensor.shape[1])

# Print the model architecture
print(model)

# DEFINE THE HINGE LOSS & OPTIMIZER
# Hinge Loss: penalizes incorrect classifications & encourages maximizing the margin.
def hinge_loss(y_true, y_pred):
    return torch.mean(torch.clamp(1 - y_true * y_pred, min=0))

# Define the optimizer SGD (Stochastic Gradient Descent)
# lr = Learning Rate (0.01) = slow learn
optimizer = optim.SGD(model.parameters(), lr=0.01)

SVM(
  (linear): Linear(in_features=4, out_features=1, bias=True)
)


In [12]:
# TRAIN THE MODEL
# Training loop, Train the Model
epochs = 100
C = 1.0  # Regularization parameter (controls the strength of L2 regularization)

for epoch in range(epochs):
    # Set the model to training mode
    model.train()

    # Zero the gradients
    optimizer.zero_grad()

    # Forward pass: Compute the model's predictions
    y_pred = model(X_train_tensor)

    # Compute the hinge loss with L2 regularization
    # C⋅∥w∥^2, hinge loss to control the magnitude of the weights
    loss = hinge_loss(y_train_tensor, y_pred) + C * torch.norm(model.linear.weight) ** 2

    # Backward pass: Compute gradients
    loss.backward()

    # Update the model parameters
    optimizer.step()

    # After every 10 Epochs current loss is printed
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 1.2154
Epoch 20, Loss: 1.0120
Epoch 30, Loss: 0.8672
Epoch 40, Loss: 0.7617
Epoch 50, Loss: 0.6827
Epoch 60, Loss: 0.6263
Epoch 70, Loss: 0.5869
Epoch 80, Loss: 0.5633
Epoch 90, Loss: 0.5515
Epoch 100, Loss: 0.5421


In [13]:
# EVALUATE THE MODEL
# Set the model to evaluation mode
model.eval()

# Disable gradient calculations for evaluation
with torch.no_grad():
    # Make predictions on the test set
    y_pred_test = model(X_test_tensor)

    # Convert predictions to binary class labels (threshold at 0)
    y_pred_labels = torch.where(y_pred_test >= 0, torch.tensor(1.0), torch.tensor(0.0))

    # Calculate accuracy
    accuracy = torch.mean((y_pred_labels == y_test_tensor).float())
    print(f"Test Accuracy: {accuracy.item() * 100:.2f}%")

Test Accuracy: 40.00%


In [14]:
# MAKE PREDICTIONS ON THE NEW DATA
# Example of new data for prediction (standardized input)
new_data = [[5.1, 3.5, 1.4, 0.2]]  # Example input
new_data_scaled = scaler.transform(new_data)  # Scale the new data

# Convert to tensor
new_data_tensor = torch.tensor(new_data_scaled, dtype=torch.float32)

# Make a prediction
with torch.no_grad():
    y_new_pred = model(new_data_tensor)
    predicted_class = torch.where(y_new_pred >= 0, torch.tensor(1.0), torch.tensor(0.0))

print(f"Predicted Class: {int(predicted_class.item())}")

Predicted Class: 1
